LAB - 1 agent_with_tools

In [ ]:
!pip install langchain-groq langchain-community wikipedia langgraph -q

import os
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.prebuilt import create_react_agent

os.environ["GROQ_API_KEY"] = "Your API Key"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

wiki_api = WikipediaAPIWrapper()

@tool
def wikipedia_tool(query: str) -> str:
    """Look up facts about people, places, events."""
    return wiki_api.run(query)

@tool
def calculator(expression: str) -> str:
    """Do arithmetic. Input a maths expression like 12*7."""
    allowed = set('0123456789+-*/(). ')
    if not set(expression).issubset(allowed):
        return 'Error: only numbers and + - * / ( ) are allowed.'
    try:
        return str(eval(expression))
    except Exception as e:
        return f'Error: {e}'

tools = [wikipedia_tool, calculator]

agent = create_react_agent(
    model=llm,
    tools=tools
)

question = ('Who developed the theory of relativity, '
            'and what is his birth year multiplied by 2?')

if __name__ == "__main__":
    print(f"Question: {question}\n")

    for step in agent.stream({'messages': [('user', question)]}):
        for node_name, node_state in step.items():
            print(f"--- {node_name.upper()} ---")
            messages = node_state.get('messages', [])
            if messages:
                print(messages[-1].content if hasattr(messages[-1], 'content') else messages[-1])

    result = agent.invoke({'messages': [('user', question)]})
    print('\nFINAL ANSWER:', result['messages'][-1].content if hasattr(result['messages'][-1], 'content') else result['messages'][-1])

LAB - 2  langgraph_chatbot

In [ ]:
!pip install langgraph langchain-groq -q

import os
from typing import TypedDict, List
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

os.environ["GROQ_API_KEY"] = "Your API Key"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

class ChatState(TypedDict):
    messages: List[str]
    name: str
    preferences: str

def chatbot(state: ChatState) -> dict:
    user_msg = state['messages'][-1]

    name = state.get('name', '')
    prefs = state.get('preferences', '')

    if 'my name is' in user_msg.lower():
        name = user_msg.lower().split('my name is')[-1].strip().title()
    if 'i like' in user_msg.lower():
        prefs = user_msg.lower().split('i like')[-1].strip()

    context = f"The user's name is {name}. They like {prefs}."
    reply = llm.invoke(context + ' Reply to: ' + user_msg).content

    return {
        'messages': state['messages'] + [reply],
        'name': name,
        'preferences': prefs
    }

builder = StateGraph(ChatState)
builder.add_node('chatbot', chatbot)
builder.add_edge(START, 'chatbot')
builder.add_edge('chatbot', END)
graph = builder.compile()

if __name__ == "__main__":
    print("Starting Chatbot... (Type 'quit' to stop)")
    state = {'messages': [], 'name': '', 'preferences': ''}

    while True:
        user = input('You: ')
        if user.lower() == 'quit':
            break
        state['messages'].append(user)
        state = graph.invoke(state)
        print('Bot:', state['messages'][-1])

LAB - 3  hitl_email_agent

In [ ]:
!pip install langchain-groq -q

import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "Your API Key"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3
)

def run_hitl_agent():
    print("--- HITL Agent Initialized ---")
    instruction = input("What would you like the email to be about? ")

    feedback = instruction
    approved = False

    while not approved:
        print("\n--- GENERATING DRAFT ---")
        draft = llm.invoke(f"Write a polite, professional email based on: {feedback}").content
        print(f"\n{draft}\n")

        decision = input("Type 'APPROVED' to finish, or type instructions to revise: ")

        if decision.strip().upper() == 'APPROVED':
            print('\n✅ Final draft approved and "sent"!')
            approved = True
        else:
            feedback = f"Original goal: {instruction}. Current draft: {draft}. Revision requested: {decision}"
            print('\n🔄 Refining draft based on your feedback...')

run_hitl_agent()

LAB - 4  chroma_rag_pdf_qa

In [ ]:
!pip install langchain langchain-community langchain-groq chromadb sentence-transformers pypdf langchain-text-splitters langchain-classic -q

import os
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

os.environ["GROQ_API_KEY"] = "Your API Key"

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded.")

pdf_name = list(uploaded.keys())[0]

pages = PyPDFLoader(pdf_name).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
db = Chroma.from_documents(chunks, embeddings)

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
qa = RetrievalQA.from_chain_type(llm=llm, retriever=db.as_retriever())

while True:
    query = input("\nAsk a question about your PDF (or type 'exit' to quit): ")
    if query.strip().lower() == 'exit':
        print("Goodbye!")
        break

    if query.strip() == "":
        continue

    response = qa.invoke(query)['result']
    print(f"\nAnswer:\n{response}")

LAB - 5  advanced_rag_pdf

In [ ]:
# =====================================================================
# STEP 1: Install required libraries (with modern updates included)
# =====================================================================
print("⏳ Installing libraries... Please wait.")
!pip install langchain langchain-community langchain-groq chromadb sentence-transformers pypdf langchain-text-splitters -q
print("✅ Libraries installed successfully!\n")

# =====================================================================
# STEP 2: Import modules and set up the LLM
# =====================================================================
import os
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq

# Set up your Groq API Key here
os.environ["GROQ_API_KEY"] = "Your API Key"

# Initialize the LLM
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# =====================================================================
# STEP 3: Upload and build the Vector Database (Same as Lab 4)
# =====================================================================
print("--- PDF UPLOAD ---")
print("Please select a short PDF file from your computer:")
uploaded = files.upload()

if not uploaded:
    raise ValueError("❌ No file uploaded. Please rerun the cell and select a PDF.")

pdf_name = list(uploaded.keys())[0]
print(f"✅ Uploaded: {pdf_name}\n")

print("⚙️ Processing PDF (Loading & Chunking)...")
pages = PyPDFLoader(pdf_name).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print(f"🔹 Total chunks created: {len(chunks)}")

print("⏳ Embedding chunks into ChromaDB...")
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
db = Chroma.from_documents(chunks, embeddings)
print("✅ Vector database ready!\n")

# =====================================================================
# STEP 4: Define RAG Helper & Self-Correction Adaptive Loop
# =====================================================================

def answer_from_pdf(query):
    # Retrieve matching document chunks
    docs = db.as_retriever().invoke(query)
    context = '\n'.join(d.page_content for d in docs)

    # Construct a strict prompt grounding the LLM
    prompt = (
        f"Use ONLY this context to answer.\n{context}\n\n"
        f"Question: {query}\n"
        "If the context does not contain the answer, reply exactly: 'I don't know'."
    )
    return llm.invoke(prompt).content

def adaptive_answer(question, max_tries=3):
    query = question
    for attempt in range(1, max_tries + 1):
        print(f'🔍 Attempt {attempt} with query: "{query}"')
        answer = answer_from_pdf(query)

        # If the model gives a valid answer (not 'I don't know'), stop and return it
        if "i don't know" not in answer.lower():
            return answer

        # Otherwise, if it hits a wall, rephrase and retry
        print("⚠️ Answer not found. Rephrasing query...")
        query = llm.invoke(f'Rephrase this search query differently to search a database: {query}').content

    return '❌ Could not find an answer after several tries.'

# =====================================================================
# STEP 5: Run the Adaptive RAG Agent
# =====================================================================
print("--- ADAPTIVE CHAT ---")
user_question = input("What is your question? (e.g., 'What is the main conclusion?')\n👉 ")

print("\n🤖 Processing Agent Workflow...")
final_result = adaptive_answer(user_question)

print(f"\n📝 Final Agent Response:\n{final_result}")

LAB - 6  autogen_writer_agent

In [ ]:
!pip install autogen -q
import autogen

config_list = [{
    'model': 'llama-3.1-8b-instant',
    'api_key': 'Your API Key',
    'base_url': 'https://api.groq.com/openai/v1'
}]

llm_config = {
    'config_list': config_list,
    'temperature': 0.4
}

writer = autogen.AssistantAgent(
    name='Writer',
    system_message='You write short, clear blog posts. When given feedback, rewrite to improve it.',
    llm_config=llm_config
)

critic = autogen.AssistantAgent(
    name='Critic',
    system_message='You review blog posts and give 2-3 specific improvements. If it is good, reply APPROVED.',
    llm_config=llm_config
)

critic.initiate_chat(
    recipient=writer,
    message='Write a 150-word blog post about the benefits of reading.',
    max_turns=4
)

LAB - 7  sqlite_memory_agent

In [ ]:
!pip install langchain-groq -q

import os
import sqlite3
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "Your API Key"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

conn = sqlite3.connect('memory.db')
conn.execute('CREATE TABLE IF NOT EXISTS facts (key TEXT, value TEXT)')
conn.commit()

def remember(key, value):
    conn.execute('INSERT INTO facts VALUES (?, ?)', (key, value))
    conn.commit()

def recall_all():
    rows = conn.execute('SELECT key, value FROM facts').fetchall()
    return '; '.join(f'{k} = {v}' for k, v in rows)

conn.execute('DELETE FROM facts')
conn.commit()

remember('name', 'Riya')
remember('favourite_subject', 'Mathematics')

memory = recall_all()
print('Recalled memory:', memory)

prompt = (
    f"Known facts: {memory}. "
    "Greet the user by name and mention their favourite subject."
)

reply = llm.invoke(prompt)
print(reply.content)

LAB - 8   8_langgraph_team_agent

In [ ]:
!pip install langgraph langchain-groq -q

import os
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

os.environ["GROQ_API_KEY"] = "Your API Key"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

class TeamState(TypedDict):
    task: str
    worker_result: str
    summary: str

def worker(state: TeamState) -> dict:
    answer = llm.invoke('Solve this math problem, show the number only: ' + state['task']).content
    return {'worker_result': answer}

def supervisor(state: TeamState) -> dict:
    summary = llm.invoke(
        f"The worker solved '{state['task']}' and got "
        f"{state['worker_result']}. Write a one-line summary."
    ).content
    return {'summary': summary}

builder = StateGraph(TeamState)
builder.add_node('worker', worker)
builder.add_node('supervisor', supervisor)
builder.add_edge(START, 'worker')
builder.add_edge('worker', 'supervisor')
builder.add_edge('supervisor', END)
graph = builder.compile()

result = graph.invoke({'task': 'What is 144 divided by 12, then plus 5?'})
print('Worker result:', result['worker_result'])
print('Supervisor summary:', result['summary'])

LAB - 9   langsmith_agent

In [ ]:
!pip install langchain langchain-groq langsmith -q

import os
from langchain_groq import ChatGroq

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_API_KEY'] = 'Your langchain API Key'
os.environ['LANGCHAIN_PROJECT'] = 'Lab9-Agentic-AI'
os.environ["GROQ_API_KEY"] = "Your API Key"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

question = (
    'A shop sells pens at 12 rupees each. '
    'If I buy 7 and pay with 100, how much change? '
    'Think step by step.'
)

reply = llm.invoke(question)
print(reply.content)

LAB - 10   duckduckgo_news_agent

In [ ]:
!pip install langchain langchain-community langchain-groq duckduckgo-search ddgs -q

import os
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchResults

os.environ["GROQ_API_KEY"] = "Your API Key"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3
)

search = DuckDuckGoSearchResults()

def search_agent(topic):
    return search.run(f'latest news about {topic}')

def summariser_agent(raw_results):
    return llm.invoke(
        'Summarise these news results into 3 short bullet points:\n' + raw_results
    ).content

def report_agent(topic, summary):
    return llm.invoke(
        f'Create a short Markdown report titled "News: {topic}". '
        f'Use a heading and these points:\n{summary}'
    ).content

topic = 'artificial intelligence'

raw = search_agent(topic)
summary = summariser_agent(raw)
report = report_agent(topic, summary)

with open('news_report.md', 'w') as f:
    f.write(report)

print(report)
print('\nSaved as news_report.md (see the Files panel on the left).')